<a href="https://colab.research.google.com/github/avikumart/DA-DS-Questions/blob/main/R_codes/neural_nets_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
install.packages("torch")
install.packages("torchvision")
library(torch)
install_torch() # Added to install additional dependencies for torch
library(torchvision)

transform <- function(x) {
    x %>%
    torch_tensor() %>%
    torch_flatten() %>%
    torch_div(255)
}

# load the cifar dataset using r
train_ds <- cifar10_dataset(
    root = "./",
    transform = transform,
    download = TRUE,
    train = TRUE
)

test_ds <- cifar10_dataset(
    root = "./",
    transform = transform,
    download = TRUE,
    train = FALSE
)

# get the led of the train and test dataset
length(train_ds)
length(test_ds$data)

In [ ]:
length(train_ds)
length(test_ds)

In [ ]:
# Create dataloaders (Batch size 64 is standard for CIFAR-10)
train_dl <- dataloader(train_ds, batch_size = 64, shuffle = TRUE)
test_dl <- dataloader(test_ds, batch_size = 64, shuffle = FALSE)

## MLP

In [ ]:
install.packages("luz") # Install the luz package if not already installed. Ideally, this would be placed with other install commands.
library(luz)            # Load the luz package to make the `fit` function available.

# 1. Define the MLP Model
# Input: 3072 (Flattened pixels), Output: 10 (Classes)
mlp_model <- nn_module(
  "MLP",
  initialize = function() {
    self$fc1 <- nn_linear(3072, 512) # First hidden layer
    self$fc2 <- nn_linear(512, 10)   # Output layer
  },
  forward = function(x) {
    x %>%
      self$fc1() %>%
      nnf_relu() %>%
      self$fc2()
  }
)

# 2. Train and Evaluate using Luz
mlp_fitted <- mlp_model %>%
  setup(
    loss = nn_cross_entropy_loss(),
    optimizer = optim_adam,
    metrics = list(luz_metric_accuracy())
  ) %>%
  set_hparams() %>%
  fit(train_dl, epochs = 10, valid_data = test_dl)

# 3. View Results
print(mlp_fitted)

## CNN

In [ ]:
# 1. Define the CNN Model
cnn_model <- nn_module(
  "CNN",
  initialize = function() {
    # Conv Layer 1: 3 input channels (RGB), 32 output channels
    self$conv1 <- nn_conv2d(in_channels = 3, out_channels = 32, kernel_size = 3)
    # Conv Layer 2: 32 inputs, 64 outputs
    self$conv2 <- nn_conv2d(in_channels = 32, out_channels = 64, kernel_size = 3)

    # Max pooling layer
    self$pool <- nn_max_pool2d(kernel_size = 2, stride = 2)

    # Fully connected layers
    # Calculation for input size:
    # 32x32 -> conv1(3x3) -> 30x30 -> pool(2x2) -> 15x15
    # 15x15 -> conv2(3x3) -> 13x13 -> pool(2x2) -> 6x6
    # Final flat size: 6 * 6 * 64 (channels) = 2304
    self$fc1 <- nn_linear(6 * 6 * 64, 128)
    self$fc2 <- nn_linear(128, 10)
  },
  forward = function(x) {
    # RESHAPE: Un-flatten the data back to (Batch, 3, 32, 32)
    x <- x$view(c(-1, 3, 32, 32))

    x %>%
      self$conv1() %>% nnf_relu() %>% self$pool() %>%
      self$conv2() %>% nnf_relu() %>% self$pool() %>%
      torch_flatten(start_dim = 2) %>%
      self$fc1() %>% nnf_relu() %>%
      self$fc2()
  }
)

# 2. Train and Evaluate using Luz
cnn_fitted <- cnn_model %>%
  setup(
    loss = nn_cross_entropy_loss(),
    optimizer = optim_adam,
    metrics = list(luz_metric_accuracy())
  ) %>%
  fit(train_dl, epochs = 10, valid_data = test_dl)

# 3. View Results
print(cnn_fitted)

## Comparison

In [ ]:
# Extract final validation accuracy
mlp_acc <- get_metrics(mlp_fitted) %>%
  filter(set == "valid") %>%
  tail(1) %>%
  pull(acc)

cnn_acc <- get_metrics(cnn_fitted) %>%
  filter(set == "valid") %>%
  tail(1) %>%
  pull(acc)

cat("MLP Accuracy:", mlp_acc, "\n")
cat("CNN Accuracy:", cnn_acc, "\n")